# Data Cleaning and Preprocessing on NYC Airbnb Dataset
**Internship Project - Oasis Infobyte (Level 1, Project 3)**  
**Author:** Aditya Halder (Data Analytics Intern)

---

### Project Overview
Data cleaning is the vital process of identifying and correcting corrupted, inaccurate, duplicate, or incomplete records from a dataset. Working with raw, "messy" data leads to skewed statistics, invalid model training, and incorrect business intelligence.

In this project, we demonstrate a complete data cleaning and preprocessing pipeline on the **New York City Airbnb Open Data (2019)** dataset. The dataset contains 48,895 entries and is a perfect real-world example with missing data, outliers, formatting inconsistencies, and multiple categorical columns.

### Preprocessing Steps Implemented:
1. **Missing Data Imputation**: Handled missing text records and numeric columns using statistical placeholders.
2. **Duplicate Removal**: Checked for and eliminated redundant transaction entries.
3. **DataType Standardization**: Converted string representations of date fields to `datetime` objects.
4. **Outlier Treatment**: Detected price/night anomalies and applied **99th percentile capping** to reduce skewness.
5. **Categorical Variable Encoding**: Applied One-Hot Encoding to categorical identifiers.
6. **Feature Scaling**: Scaled continuous variables using Scikit-Learn's `StandardScaler` to prepare the data for downstream machine learning.



## 1. Setup and Libraries
Importing required modules for wrangling, math operations, plotting, and feature scaling.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# Set visualization style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.family'] = 'sans-serif'

print("Libraries imported successfully!")


## 2. Data Loading
Loading the raw Airbnb dataset using Pandas with relative paths.


In [ ]:
# Load the dataset
file_path = '../Dataset/Dataset 1/AB_NYC_2019.csv'
df = pd.read_csv(file_path)

# Display raw shape and head
print("Raw Dataset Dimensions:", df.shape)
df.head()


## 3. Identify and Handle Missing Values
Scanning the dataset for missing/null values per column. We isolate columns that have missing data to study their distribution.


In [ ]:
# Check for missing values in each column
missing_counts = df.isnull().sum()
print("Missing entries per column:\n", missing_counts[missing_counts > 0])


### Plot Missing Values Before Cleaning


In [ ]:
plt.figure(figsize=(10, 6))
missing_before = df.isnull().sum()
missing_before = missing_before[missing_before > 0].sort_values(ascending=False)

ax = sns.barplot(x=missing_before.values, y=missing_before.index, palette="flare_r")
plt.title('Missing Values Count Before Data Cleaning', pad=20, weight='bold', fontsize=15)
plt.xlabel('Number of Missing Entries', labelpad=10)
plt.ylabel('Features / Columns', labelpad=10)

# Add annotations
for i, v in enumerate(missing_before.values):
    pct = (v / len(df)) * 100
    ax.text(v + 100, i, f"{v:,} ({pct:.2f}%)", va='center', weight='bold', fontsize=10)

plt.xlim(0, max(missing_before.values) * 1.18)
plt.tight_layout()
plt.savefig('../Visualizations/missing_values_before.png', dpi=300)
plt.show()


### Impute Missing Entries
- We fill null `name` and `host_name` text descriptions with `'Unknown'`.
- Missing reviews columns (`reviews_per_month`) are filled with `0.0` (indicating no activity).
- We convert `last_review` to standard datetime format and fill NaT values with a sentinel date `'1970-01-01'` to represent 'Not Reviewed' in a consistent datetype format.


In [ ]:
# Fill missing names and host names
df['name'] = df['name'].fillna('Unknown')
df['host_name'] = df['host_name'].fillna('Unknown')

# Fill missing reviews per month with 0.0
df['reviews_per_month'] = df['reviews_per_month'].fillna(0.0)

# Convert last_review to datetime and fill with sentinel date
df['last_review'] = pd.to_datetime(df['last_review'])
df['last_review'] = df['last_review'].fillna(pd.Timestamp('1970-01-01'))

print("Imputations completed.")


## 4. Duplicate Records Check
Scanning and removing duplicate transactions from the dataset.


In [ ]:
# Check duplicate entries
duplicates = df.duplicated().sum()
print(f"Number of duplicate records found: {duplicates}")

# Drop duplicates if any exist
if duplicates > 0:
    df = df.drop_duplicates()
    print("Duplicates dropped successfully.")


## 5. Outlier Detection and Treatment
Extreme value analysis for price and minimum nights. We first clean invalid listings (price = 0) and check the statistical IQR ranges.


In [ ]:
# Filter out invalid prices (listings with price = 0)
invalid_prices_count = (df['price'] == 0).sum()
print(f"Number of listings with Price = $0: {invalid_prices_count}")

df = df[df['price'] > 0]


In [ ]:
# Price IQR outlier analysis
Q1 = df['price'].quantile(0.25)
Q3 = df['price'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Price IQR Bounds: Lower = {lower_bound:.2f}, Upper = {upper_bound:.2f}")
outliers = ((df['price'] < lower_bound) | (df['price'] > upper_bound)).sum()
print(f"Number of outliers outside IQR bounds: {outliers} ({outliers/len(df)*100:.2f}%)")


### Capping Extreme Outliers
To preserve sample size and represent realistic listings, we apply **99th percentile capping** for both `price` and `minimum_nights` rather than dropping rows.


In [ ]:
# Compute 99th percentile capping values
price_cap = df['price'].quantile(0.99)
nights_cap = df['minimum_nights'].quantile(0.99)

print(f"99th percentile price capping value: ${price_cap:.2f}")
print(f"99th percentile minimum nights capping value: {nights_cap:.2f} nights")

# Cap values
df['price_cleaned'] = df['price'].clip(upper=price_cap)
df['minimum_nights_cleaned'] = df['minimum_nights'].clip(upper=nights_cap)


### Box Plot Comparisons of Raw vs. Outlier-Treated Columns


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Price boxplots
sns.boxplot(data=df, x='price', ax=axes[0, 0], color='#fc8181', flierprops=dict(markersize=3))
axes[0, 0].set_title('Raw Price Distribution (With Outliers)', weight='bold')
axes[0, 0].set_xlabel('Price ($)')

sns.boxplot(data=df, x='price_cleaned', ax=axes[0, 1], color='#68d391')
axes[0, 1].set_title('Cleaned Price Distribution (Capped at 99th Percentile)', weight='bold')
axes[0, 1].set_xlabel('Price ($)')

# Minimum Nights boxplots
sns.boxplot(data=df, x='minimum_nights', ax=axes[1, 0], color='#fc8181', flierprops=dict(markersize=3))
axes[1, 0].set_title('Raw Minimum Nights (With Outliers)', weight='bold')
axes[1, 0].set_xlabel('Nights')

sns.boxplot(data=df, x='minimum_nights_cleaned', ax=axes[1, 1], color='#68d391')
axes[1, 1].set_title('Cleaned Minimum Nights (Capped at 99th Percentile)', weight='bold')
axes[1, 1].set_xlabel('Nights')

plt.suptitle('Outlier Detection & Capping Comparison', fontsize=16, weight='bold', y=0.98)
plt.tight_layout()
plt.savefig('../Visualizations/outlier_detection.png', dpi=300)
plt.show()


## 6. Categorical Variable Encoding
We encode the categorical features `neighbourhood_group` and `room_type` using One-Hot Encoding (`pd.get_dummies`) to convert them to numerical format suitable for standard regression and classification algorithms.


In [ ]:
# One-hot encode categorical variables
encoded_df = pd.get_dummies(df, columns=['neighbourhood_group', 'room_type'], drop_first=True)
print("Dimensions after One-Hot Encoding:", encoded_df.shape)
encoded_df.head()


## 7. Numerical Feature Scaling
Continuous attributes like Price, Minimum Nights, Review counts, and Availability have different numerical ranges. Standardizing them using `StandardScaler` sets their mean to 0 and variance to 1, preventing high-valued variables from dominating downstream calculations.


In [ ]:
# Columns to scale
scale_cols = ['price_cleaned', 'minimum_nights_cleaned', 'number_of_reviews', 'reviews_per_month', 'availability_365']

# Fit and transform
scaler = StandardScaler()
scaled_features = scaler.fit_transform(encoded_df[scale_cols])

# Add scaled columns back to dataframe
for idx, col in enumerate(scale_cols):
    encoded_df[col + '_scaled'] = scaled_features[:, idx]

print("Feature scaling completed.")
encoded_df[[col + '_scaled' for col in scale_cols]].describe()


## 8. Verifying Preprocessed Data
Let's verify that no missing values remain in the cleaned data frame and view feature distributions and correlations.


In [ ]:
# Confirm zero missing values
plt.figure(figsize=(10, 4))
missing_after = df.isnull().sum()

sns.barplot(x=missing_after.index, y=missing_after.values, color='#48bb78')
plt.title('Missing Values Count After Preprocessing & Imputation', pad=20, weight='bold', fontsize=15)
plt.xlabel('Features / Columns', labelpad=10)
plt.ylabel('Missing Entries Count')
plt.xticks(rotation=45, ha='right')
plt.ylim(0, 5) 
plt.tight_layout()
plt.savefig('../Visualizations/missing_values_after.png', dpi=300)
plt.show()


In [ ]:
# Feature distributions histograms
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.histplot(data=df, x='price_cleaned', kde=True, ax=axes[0], color='#4a5568', bins=35)
axes[0].set_title('Cleaned Price Distribution Histogram', weight='bold')
axes[0].set_xlabel('Price ($)')

sns.histplot(data=df, x='availability_365', kde=True, ax=axes[1], color='#3182ce', bins=35)
axes[1].set_title('Availability 365 Distribution Histogram', weight='bold')
axes[1].set_xlabel('Days Available Per Year')

plt.tight_layout()
plt.savefig('../Visualizations/feature_distribution.png', dpi=300)
plt.show()


In [ ]:
# Correlation Heatmap
plt.figure(figsize=(10, 8))
cols = ['price_cleaned', 'minimum_nights_cleaned', 'number_of_reviews', 'reviews_per_month', 'availability_365', 'calculated_host_listings_count']
corr_matrix = df[cols].corr()

sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".3f", linewidths=.5, 
            square=True, cbar_kws={"shrink": .8}, annot_kws={'size': 12, 'weight': 'bold'})
plt.title('Correlation Matrix of Cleaned Continuous Features', pad=20, weight='bold', fontsize=15)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('../Visualizations/correlation_heatmap.png', dpi=300)
plt.show()


## 9. Conclusion & Business Takeaways

Summary of data preprocessing transformations:
1. **Dimensions**: The raw dataset contains **48,895 listings**. After removing invalid listings (price = 0), the dataset contains **48,884 listings**.
2. **Missing Entries**: Successfully cleaned **16 missing name records**, **21 host name records**, and **10,052 reviews/date metrics** using imputation. The dataset contains **0 missing entries**.
3. **Outliers Treatment**: Capped extreme values in price ($799 capping limit) and minimum nights (45 nights capping limit) to normalize the distributions.
4. **Model Readiness**: The final preprocessed data frame includes 22 columns containing scaled numerical variables and dummy variables, ready for machine learning.

---
**End of Project**

